<a href="https://colab.research.google.com/github/poggerssLL/Automatica---Grupo-2/blob/main/etapa-01-logica/08_Base_de_Conhecimento_e_Regras_de_Diagnostico.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Aula 08 - Notebook: Sistemas Especialistas — Base de Conhecimento e Diagnóstico em SCADA

## 1. Fundamentos Matemáticos: Arquitetura de Sistemas Baseados em Regras (RBS)

Um Sistema Especialista Baseado em Regras para automação industrial de tempo real é formalmente modelado pela tripla:

$$\langle \mathcal{F}, \mathcal{R}, \mathcal{E} \rangle$$

Onde:
1. **$\mathcal{F}$ (Base de Fatos Dinâmica):** Conjunto finito de proposições que refletem a telemetria instantânea da máquina de envasamento e fatos derivados:
   $$\mathcal{F}(t) = \{f_1, f_2, \dots, f_m\} \subseteq \mathcal{U}_{\text{fatos}}$$
2. **$\mathcal{R}$ (Base de Conhecimento):** Conjunto de regras de produção expressas em **Cláusulas de Horn Definidas**:
   $$R_i: \quad \text{SE } (A_{i,1} \land A_{i,2} \land \dots \land A_{i,k}) \quad \text{ENTÃO } \quad C_i$$
3. **$\mathcal{E}$ (Estratégia de Resolução de Conflitos e Encadeamento):** Arbitragem baseada em prioridade de segurança (IEC 61508 / SIL) executando inferência por **Forward Chaining** (encadeamento para frente até saturação).

---

## 2. Aplicação na Máquina de Envasamento de Copos Plásticos (UNIFEI)

Neste notebook implementamos:
1. A classe `Fato` para modelagem de sinais do SCADA, alarmes e conclusões inferidas.
2. A classe `RegraDiagnostico` parametrizada com antecedentes, consequente, prioridade SIL, severidade e Procedimento Operacional Padrão (POP).
3. A classe `BaseConhecimentoSCADA` com motor de inferência forward-chaining, resolução de conflitos e validação de consistência.
4. Simulação de múltiplos cenários industriais críticos: colisão mecânica, falha térmica de termosselagem, desabastecimento de copos e falha de vácuo no pick-and-place.


In [1]:
from typing import Dict, List, Set, Any, Tuple, Optional
from dataclasses import dataclass, field
import time

def formatar_tabela(dados: List[Dict[str, Any]]) -> str:
    """Formata lista de dicionarios em tabela ASCII pura."""
    if not dados:
        return "Tabela Vazia"
    colunas = list(dados[0].keys())
    larguras = {c: len(str(c)) for c in colunas}
    for row in dados:
        for c in colunas:
            larguras[c] = max(larguras[c], len(str(row.get(c, ""))))
    header = " | ".join(f"{c:<{larguras[c]}}" for c in colunas)
    divisor = "-+-".join("-" * larguras[c] for c in colunas)
    linhas = [header, divisor]
    for row in dados:
        linhas.append(" | ".join(f"{str(row.get(c, '')):<{larguras[c]}}" for c in colunas))
    return "\n".join(linhas)

# ==============================================================================
# 1. MODELAGEM DE FATOS E REGRAS DE DIAGNÓSTICO (CLÁUSULAS DE HORN)
# ==============================================================================

@dataclass
class Fato:
    nome: str
    valor: bool
    origem: str = "Telemetria"
    timestamp: float = field(default_factory=time.time)
    descricao: str = ""

@dataclass
class RegraDiagnostico:
    id_regra: str
    antecedentes: List[str]
    consequente: str
    diagnostico: str
    severidade: str
    prioridade: int
    pop: str

    def disparavel(self, fatos_ativos: Set[str]) -> bool:
        """Verifica se todos os antecedentes da Cláusula de Horn estão satisfeitos."""
        return all(ant in fatos_ativos for ant in self.antecedentes)

# ==============================================================================
# 2. MOTOR DE INFERÊNCIA E BASE DE CONHECIMENTO SCADA
# ==============================================================================

class BaseConhecimentoSCADA:
    def __init__(self):
        self.regras: List[RegraDiagnostico] = []
        self.fatos_dinamicos: Dict[str, Fato] = {}
        self.inicializar_catalogo_regras_envasadora()

    def inicializar_catalogo_regras_envasadora(self):
        """Catálogo especialista de falhas e diagnósticos da Máquina de Envase."""
        self.regras = [
            RegraDiagnostico(
                id_regra="R-01",
                antecedentes=["SOLICITA_GIRO_MESA", "PRENSA_NAO_RECUADA"],
                consequente="TRIP_COLISAO_MESA",
                diagnostico="Risco Iminente de Colisão Mecânica: Mesa indexando com Prensa Térmica Avançada",
                severidade="EMERGÊNCIA",
                prioridade=10,
                pop="POP-M01: Bloquear imediatamente o motor da mesa M-001. Comandar recuo forçado de F e emitir alarme audiovisual."
            ),
            RegraDiagnostico(
                id_regra="R-02",
                antecedentes=["SOLICITA_PRENSA", "TEMPERATURA_ABAIXO_MIN"],
                consequente="BLOQUEIO_SELAGEM_FRIO",
                diagnostico="Temperatura Insuficiente no Cabeçote Térmico (< 180°C)",
                severidade="ALTA",
                prioridade=8,
                pop="POP-S04: Inibir descida da prensa F. Verificar resistência HT-401 e sensor termopar TIT-401."
            ),
            RegraDiagnostico(
                id_regra="R-03",
                antecedentes=["SOLICITA_GIRO_BRACO", "VACUO_NAO_CONFIRMADO"],
                consequente="FALHA_CAPTURA_TAMPA",
                diagnostico="Perda de Sucção / Falta de Tampa no Manipulador Pick-and-Place",
                severidade="MÉDIA",
                prioridade=6,
                pop="POP-P03: Pausar braço giratório XV-301. Inspecionar ventosa de vácuo VAC-301 e magazine de tampas."
            ),
            RegraDiagnostico(
                id_regra="R-04",
                antecedentes=["CICLO_DISPENSA_CONCLUIDO", "COPO_ESTACAO1_AUSENTE"],
                consequente="MAGAZINE_COPOS_VAZIO",
                diagnostico="Desabastecimento ou Trancamento de Copos Plásticos no Setor 100",
                severidade="ALTA",
                prioridade=7,
                pop="POP-D01: Parar ciclo contínuo da mesa. Reabastecer magazine e inspecionar sensor ZS-102."
            ),
            RegraDiagnostico(
                id_regra="R-05",
                antecedentes=["SOLICITA_DOSE_AGUA", "BICO_FECHADO"],
                consequente="SOBREPRESSAO_DOSADOR",
                diagnostico="Risco de Ruptura Hidráulica por Injeção de Água com Válvula de Bico Fechada",
                severidade="CRÍTICA",
                prioridade=9,
                pop="POP-E02: Abortar avanço do dosador C imediatamente. Comandar abertura de emergência do bico XV-203."
            ),
            RegraDiagnostico(
                id_regra="R-06",
                antecedentes=["FIM_CURSO_AVANCO_ATIVO", "FIM_CURSO_RECUO_ATIVO"],
                consequente="FALHA_INCOERENCIA_SENSOR",
                diagnostico="Incoerência Elétrica: Leitura simultânea antagônica de avanço e recuo no mesmo atuador",
                severidade="ALTA",
                prioridade=8,
                pop="POP-I01: Inibir modo automático. Sinalizar à manutenção curto-circuito no par de sensores magnéticos."
            ),
            RegraDiagnostico(
                id_regra="R-07",
                antecedentes=["TRIP_COLISAO_MESA"],
                consequente="ALARME_GERAL_PARADA",
                diagnostico="Parada Total de Segurança da Planta por Trip de Emergência",
                severidade="EMERGÊNCIA",
                prioridade=10,
                pop="POP-G00: Desarmar contatores de potência e acionar sinalizador estroboscópico/sonoro."
            )
        ]

    def adicionar_fato(self, nome: str, valor: bool = True, origem: str = "Telemetria", descricao: str = ""):
        if valor:
            self.fatos_dinamicos[nome] = Fato(nome=nome, valor=valor, origem=origem, descricao=descricao)
        elif nome in self.fatos_dinamicos:
            del self.fatos_dinamicos[nome]

    def limpar_fatos(self):
        self.fatos_dinamicos.clear()

    def executar_inferencia_forward_chaining(self) -> List[Dict[str, Any]]:
        """
        Motor de Encadeamento para Frente (Forward Chaining) com Resolução de Conflitos por Prioridade SIL.
        Executa sucessivos passos de inferência até a saturação da base de fatos.
        """
        fatos_ativos = set(self.fatos_dinamicos.keys())
        regras_disparadas = []
        novas_inferencias = True

        while novas_inferencias:
            novas_inferencias = False
            candidatas = [r for r in self.regras if r.disparavel(fatos_ativos) and r.consequente not in fatos_ativos]

            if candidatas:
                # Estratégia de Resolução: Ordenar por Prioridade Decrescente
                candidatas.sort(key=lambda r: r.prioridade, reverse=True)
                regra_escolhida = candidatas[0]

                # Disparo da regra selecionada
                fatos_ativos.add(regra_escolhida.consequente)
                self.adicionar_fato(
                    nome=regra_escolhida.consequente,
                    valor=True,
                    origem=f"Inferência ({regra_escolhida.id_regra})",
                    descricao=regra_escolhida.diagnostico
                )
                regras_disparadas.append({
                    "Regra": regra_escolhida.id_regra,
                    "Prioridade": regra_escolhida.prioridade,
                    "Severidade": regra_escolhida.severidade,
                    "Diagnóstico Causa-Raiz": regra_escolhida.diagnostico,
                    "Fato Inferido": regra_escolhida.consequente,
                    "Ação Prescritiva (POP)": regra_escolhida.pop
                })
                novas_inferencias = True

        return regras_disparadas

    def obter_relatorio_regras(self) -> List[Dict[str, Any]]:
        return [
            {
                "ID": r.id_regra,
                "Antecedentes": " AND ".join(r.antecedentes),
                "Consequente": r.consequente,
                "Severidade": r.severidade,
                "Prio": r.prioridade
            }
            for r in self.regras
        ]

# ==============================================================================
# 3. BATERIA DE TESTES E SIMULAÇÃO DE CENÁRIOS INDUSTRIAIS
# ==============================================================================

kb = BaseConhecimentoSCADA()

print("=======================================================================")
print("CATÁLOGO DE REGRAS ESPECIALISTAS DE DIAGNÓSTICO (ENVASE UNIFEI)")
print("=======================================================================")
print(formatar_tabela(kb.obter_relatorio_regras()))

print("\n=======================================================================")
print("CENÁRIO 1: RISCO DE COLISÃO MECÂNICA E DISPARO EM CASCATA (FORWARD CHAINING)")
print("=======================================================================")
kb.limpar_fatos()
kb.adicionar_fato("SOLICITA_GIRO_MESA", True, origem="FSM Mesa", descricao="Comando de indexação")
kb.adicionar_fato("PRENSA_NAO_RECUADA", True, origem="Sensor ZSC-401", descricao="Prensa F ainda atuada")

diag1 = kb.executar_inferencia_forward_chaining()
print(formatar_tabela(diag1))

assert len(diag1) == 2
assert diag1[0]["Regra"] == "R-01"
assert diag1[1]["Regra"] == "R-07"
assert "ALARME_GERAL_PARADA" in kb.fatos_dinamicos

print("\n=======================================================================")
print("CENÁRIO 2: FALHA TÉRMICA DE TERMOSSOLAGEM E RISCO DE SOBREPRESSÃO HIDRÁULICA")
print("=======================================================================")
kb.limpar_fatos()
kb.adicionar_fato("SOLICITA_PRENSA", True, origem="FSM Setor 400")
kb.adicionar_fato("TEMPERATURA_ABAIXO_MIN", True, origem="Transmissor TIT-401")
kb.adicionar_fato("SOLICITA_DOSE_AGUA", True, origem="FSM Setor 200")
kb.adicionar_fato("BICO_FECHADO", True, origem="Sensor ZSC-203")

diag2 = kb.executar_inferencia_forward_chaining()
print(formatar_tabela(diag2))

# Validação da Arbitragem por Prioridade: R-05 (Prio 9) deve disparar antes de R-02 (Prio 8)
assert diag2[0]["Regra"] == "R-05"
assert diag2[1]["Regra"] == "R-02"

print("\n=======================================================================")
print("CENÁRIO 3: DESABASTECIMENTO DE INSUMO (MAGAZINE DE COPOS VAZIO)")
print("=======================================================================")
kb.limpar_fatos()
kb.adicionar_fato("CICLO_DISPENSA_CONCLUIDO", True, origem="FSM Setor 100")
kb.adicionar_fato("COPO_ESTACAO1_AUSENTE", True, origem="Sensor ZS-102")

diag3 = kb.executar_inferencia_forward_chaining()
print(formatar_tabela(diag3))

assert len(diag3) == 1
assert diag3[0]["Regra"] == "R-04"

print("\n[OK] Laboratório 08 validado com 100% de sucesso formal, encadeamento e prescrição!")


CATÁLOGO DE REGRAS ESPECIALISTAS DE DIAGNÓSTICO (ENVASE UNIFEI)
ID   | Antecedentes                                       | Consequente              | Severidade | Prio
-----+----------------------------------------------------+--------------------------+------------+-----
R-01 | SOLICITA_GIRO_MESA AND PRENSA_NAO_RECUADA          | TRIP_COLISAO_MESA        | EMERGÊNCIA | 10  
R-02 | SOLICITA_PRENSA AND TEMPERATURA_ABAIXO_MIN         | BLOQUEIO_SELAGEM_FRIO    | ALTA       | 8   
R-03 | SOLICITA_GIRO_BRACO AND VACUO_NAO_CONFIRMADO       | FALHA_CAPTURA_TAMPA      | MÉDIA      | 6   
R-04 | CICLO_DISPENSA_CONCLUIDO AND COPO_ESTACAO1_AUSENTE | MAGAZINE_COPOS_VAZIO     | ALTA       | 7   
R-05 | SOLICITA_DOSE_AGUA AND BICO_FECHADO                | SOBREPRESSAO_DOSADOR     | CRÍTICA    | 9   
R-06 | FIM_CURSO_AVANCO_ATIVO AND FIM_CURSO_RECUO_ATIVO   | FALHA_INCOERENCIA_SENSOR | ALTA       | 8   
R-07 | TRIP_COLISAO_MESA                                  | ALARME_GERAL_PARADA      | EMERGÊNCI